# Metagenomic Risk Score (MRS) for Water: Template Notebook

This notebook provides a practical framework to compute a composite Metagenomic Risk Score (MRS) for water samples, based on:

- Pathogen abundance and severity
- Antimicrobial resistance (AMR) gene abundance and clinical importance
- Optional co-occurrence/context (e.g., ARGs on plasmids or within pathogen bins)

It supports both absolute abundances (e.g., copies or genomes per liter) and relative metrics (e.g., TPM/RPKM). You can run it as-is with the built-in synthetic demo or point it to your CSV files.

Outputs include:
- Per-sample Pathogen score, AMR score, Interaction bonus, Raw composite score
- Scaled 0–100 risk index and categorical banding
- Plots and CSV exports of results and contributors

Please read the Quick Start below for expected input formats and how to customize weights and scaling.

## Quick Start

1) Prepare input CSVs (UTF-8). You can omit optional columns; defaults will be used. Units must be consistent across samples.

- pathogens.csv (per sample per pathogen):
  - Required: sample_id, pathogen, abundance
  - Optional: unit, S_p (severity 1–5), H_p (human pathogenicity 0.5–2), V_p (viability 0.5–1), viability (True/False)

- args.csv (per sample per ARG):
  - Required: sample_id, arg, abundance
  - Optional: unit, C_g (clinical importance 1–5), M_g (mobility 1–2), H_g (host-range 1–2)

- cooccurrence.csv (optional, per sample per pathogen–ARG pair):
  - Columns: sample_id, pathogen, arg
  - Optional: linked_abundance (if known), on_plasmid (True/False), on_pathogen (True/False), B_pg (co-occurrence multiplier, e.g., 1.5–3)

2) Put your files in the same directory as this notebook or update the file paths in the next cells.

3) Customize weights in the Config cell (severity for pathogens, clinical importance for ARGs, exposure scenario, etc.).

4) Run the notebook. It will compute:
- S_path = sum over pathogens (A_p × W_p)
- S_amr = sum over ARGs (A_g × W_g)
- S_int = sum over pathogen–ARG links (A_link × (B_pg − 1))
- S_raw = S_path + S_amr + S_int
- Scaled score 0–100 via percentile scaling (default) and a saturating alternative.

5) Review the figures and exported CSVs in the outputs/ directory.

In [ ]:
# Imports and settings
import os
import math
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pd.options.display.max_rows = 200
pd.options.display.max_columns = 200
sns.set(style="whitegrid", context="talk")

# If you run into missing packages, you can try installing via pip (uncomment):
# %pip install pandas numpy seaborn matplotlib scipy


## Configuration: weights, exposure, and scoring

Edit the mappings below to reflect your program's priorities. Defaults are sensible but conservative placeholders. If your input tables already contain per-row S_p/H_p/V_p or C_g/M_g/H_g values, they will override these defaults.

In [ ]:
# Exposure scenario multiplier (e.g., 1 for drinking, 0.5 for recreational)
EXPOSURE_E = 1.0

# Default pathogen weights by name (case-insensitive match after stripping)
# Each entry: {"S_p": clinical severity 1–5, "H_p": human pathogenicity 0.5–2}
PATHOGEN_DEFAULT_WEIGHTS = {
    "campylobacter jejuni": {"S_p": 4.0, "H_p": 2.0},
    "salmonella enterica": {"S_p": 5.0, "H_p": 2.0},
    "shigella sonnei": {"S_p": 5.0, "H_p": 2.0},
    "escherichia coli o157": {"S_p": 5.0, "H_p": 2.0},
    "vibrio cholerae": {"S_p": 5.0, "H_p": 2.0},
    "norovirus": {"S_p": 4.0, "H_p": 2.0},
}

# Default ARG weights by gene symbol (case-insensitive match after stripping)
# Each entry: {"C_g": clinical importance 1–5, "M_g": mobility 1–2, "H_g": host-range 1–2}
ARG_DEFAULT_WEIGHTS = {
    "blandm": {"C_g": 5.0, "M_g": 2.0, "H_g": 2.0},
    "mcr-1": {"C_g": 5.0, "M_g": 2.0, "H_g": 2.0},
    "blakpc": {"C_g": 5.0, "M_g": 2.0, "H_g": 2.0},
    "blactx-m": {"C_g": 4.0, "M_g": 2.0, "H_g": 1.5},
    "qnrS": {"C_g": 3.0, "M_g": 2.0, "H_g": 1.5},
}

# Viability assumption if not provided: 1.0 (DNA detected). If you use PMA/rRNA evidence, set per-row V_p.
DEFAULT_VIABILITY_VP = 1.0

# Co-occurrence baseline multiplier if evidence exists but B_pg not provided
DEFAULT_B_PG = 2.0

# Categorical risk bands for the 0–100 scaled score
RISK_BANDS = [0, 20, 40, 70, 100]
RISK_LABELS = ["Low", "Moderate", "Elevated", "High"]


## Helper functions

In [ ]:
def _norm_key(x):
    if pd.isna(x):
        return None
    return str(x).strip().lower()


def load_table(path, required_cols):
    """Load a CSV/TSV with flexible delimiter. Ensures required columns exist.
    Returns empty DataFrame if path is None or file missing.
    """
    if not path or not os.path.exists(path):
        return pd.DataFrame(columns=required_cols)
    # Try CSV then TSV
    try:
        df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path, sep='\t')
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}")
    return df


def merge_pathogen_weights(df_path):
    df = df_path.copy()
    for col in ["sample_id", "pathogen", "abundance"]:
        if col not in df.columns:
            raise ValueError(f"Pathogens table missing required column: {col}")
    # Normalize keys for merging defaults
    df["_pkey"] = df["pathogen"].apply(_norm_key)
    # Attach defaults where not provided
    df["S_p"] = df.get("S_p", pd.Series([np.nan]*len(df)))
    df["H_p"] = df.get("H_p", pd.Series([np.nan]*len(df)))
    df["V_p"] = df.get("V_p", pd.Series([np.nan]*len(df)))
    
    def_vals = df["_pkey"].map({k: v.get("S_p") for k, v in PATHOGEN_DEFAULT_WEIGHTS.items()})
    df.loc[df["S_p"].isna(), "S_p"] = def_vals[df["S_p"].isna()].astype(float)
    def_vals = df["_pkey"].map({k: v.get("H_p") for k, v in PATHOGEN_DEFAULT_WEIGHTS.items()})
    df.loc[df["H_p"].isna(), "H_p"] = def_vals[df["H_p"].isna()].astype(float)

    # Viability: prefer explicit V_p; else map from 'viability' boolean; else default
    if "viability" in df.columns:
        viab = df["viability"].astype(str).str.lower().isin(["true", "1", "yes", "y"]).astype(float)
        # Map True->1.0, False->0.5 as a gentle downweight if DNA-only
        viab = viab.map({1.0: 1.0, 0.0: 0.5})
        df.loc[df["V_p"].isna(), "V_p"] = viab[df["V_p"].isna()]
    df["V_p"] = df["V_p"].fillna(DEFAULT_VIABILITY_VP)

    # Fill any remaining missing weights with conservative defaults
    df["S_p"] = df["S_p"].fillna(3.0)
    df["H_p"] = df["H_p"].fillna(1.0)

    # Compute weight
    df["W_p"] = df["S_p"].astype(float) * df["H_p"].astype(float) * float(EXPOSURE_E) * df["V_p"].astype(float)
    df["pathogen_contrib"] = df["abundance"].astype(float) * df["W_p"]
    return df


def merge_arg_weights(df_arg):
    df = df_arg.copy()
    for col in ["sample_id", "arg", "abundance"]:
        if col not in df.columns:
            raise ValueError(f"ARGs table missing required column: {col}")
    df["_gkey"] = df["arg"].apply(_norm_key)
    df["C_g"] = df.get("C_g", pd.Series([np.nan]*len(df)))
    df["M_g"] = df.get("M_g", pd.Series([np.nan]*len(df)))
    df["H_g"] = df.get("H_g", pd.Series([np.nan]*len(df)))

    defC = df["_gkey"].map({k: v.get("C_g") for k, v in ARG_DEFAULT_WEIGHTS.items()})
    defM = df["_gkey"].map({k: v.get("M_g") for k, v in ARG_DEFAULT_WEIGHTS.items()})
    defH = df["_gkey"].map({k: v.get("H_g") for k, v in ARG_DEFAULT_WEIGHTS.items()})

    df.loc[df["C_g"].isna(), "C_g"] = defC[df["C_g"].isna()].astype(float)
    df.loc[df["M_g"].isna(), "M_g"] = defM[df["M_g"].isna()].astype(float)
    df.loc[df["H_g"].isna(), "H_g"] = defH[df["H_g"].isna()].astype(float)

    df["C_g"] = df["C_g"].fillna(2.0)
    df["M_g"] = df["M_g"].fillna(1.0)
    df["H_g"] = df["H_g"].fillna(1.0)

    df["W_g"] = df["C_g"].astype(float) * df["M_g"].astype(float) * df["H_g"].astype(float)
    df["arg_contrib"] = df["abundance"].astype(float) * df["W_g"]
    return df


def compute_interaction(df_links, df_p=None, df_g=None):
    """Compute S_int from optional co-occurrence links.
    df_links must have: sample_id, pathogen, arg; optional: linked_abundance, B_pg, on_plasmid, on_pathogen.
    If linked_abundance missing, approximate as min(A_pathogen, A_ARG) for that sample.
    If B_pg missing, set to DEFAULT_B_PG or increase to 2.5 if on_plasmid or on_pathogen is True.
    Returns per-link contributions and per-sample S_int.
    """
    if df_links is None or df_links.empty:
        return pd.DataFrame(columns=["sample_id", "pathogen", "arg", "A_link", "B_pg", "I_pg"]), pd.DataFrame(columns=["sample_id", "S_int"]) 

    links = df_links.copy()
    for col in ["sample_id", "pathogen", "arg"]:
        if col not in links.columns:
            raise ValueError(f"Co-occurrence table missing required column: {col}")
    links["B_pg"] = links.get("B_pg", pd.Series([np.nan]*len(links)))

    # Approximate A_link if not provided
    if "linked_abundance" not in links.columns or links["linked_abundance"].isna().all():
        if df_p is None or df_g is None:
            # No way to approximate without parent tables
            links["A_link"] = np.nan
        else:
            p_map = (df_p.groupby(["sample_id", "pathogen"])['abundance']
                        .first().rename("A_p")).reset_index()
            g_map = (df_g.groupby(["sample_id", "arg"])['abundance']
                        .first().rename("A_g")).reset_index()
            links = links.merge(p_map, on=["sample_id", "pathogen"], how="left")
            links = links.merge(g_map, on=["sample_id", "arg"], how="left")
            links["A_link"] = links[["A_p", "A_g"]].min(axis=1)
    else:
        links["A_link"] = links["linked_abundance"].astype(float)

    # Determine B_pg if missing
    if "on_plasmid" in links.columns or "on_pathogen" in links.columns:
        on_plasmid = links.get("on_plasmid", False).astype(str).str.lower().isin(["true", "1", "yes", "y"]).astype(bool)
        on_pathogen = links.get("on_pathogen", False).astype(str).str.lower().isin(["true", "1", "yes", "y"]).astype(bool)
        suggest = np.where(on_plasmid | on_pathogen, np.maximum(DEFAULT_B_PG, 2.5), DEFAULT_B_PG)
        links.loc[links["B_pg"].isna(), "B_pg"] = suggest[links["B_pg"].isna()]
    links["B_pg"] = links["B_pg"].fillna(DEFAULT_B_PG)

    # Compute interaction increment I_pg = A_link * (B_pg - 1)
    links["I_pg"] = links["A_link"].astype(float) * (links["B_pg"].astype(float) - 1.0)
    S_int = links.groupby("sample_id")["I_pg"].sum().rename("S_int").reset_index()
    return links, S_int


def percentile_scale(series, lo=5, hi=95):
    p_lo = np.nanpercentile(series, lo) if len(series) else 0.0
    p_hi = np.nanpercentile(series, hi) if len(series) else 1.0
    if not np.isfinite(p_lo):
        p_lo = 0.0
    if not np.isfinite(p_hi) or p_hi == p_lo:
        p_hi = p_lo + 1e-9
    x = (series - p_lo) / (p_hi - p_lo)
    return (100 * np.clip(x, 0, 1)).astype(float), p_lo, p_hi


def saturating_scale(series, alpha=None):
    # Score = 100 * (1 - exp(-alpha * x)), alpha chosen so median maps to ~50 if not specified
    x = series.astype(float)
    if alpha is None:
        med = np.nanmedian(x) if len(x) else 1.0
        if med <= 0:
            alpha = 1.0
        else:
            alpha = math.log(2) / med
    y = 100 * (1 - np.exp(-alpha * x))
    return y.astype(float), alpha


def compute_scores(df_pathogens, df_args, df_links=None):
    # Merge weights and contributions
    P = merge_pathogen_weights(df_pathogens) if not df_pathogens.empty else pd.DataFrame(columns=["sample_id", "pathogen_contrib"])
    G = merge_arg_weights(df_args) if not df_args.empty else pd.DataFrame(columns=["sample_id", "arg_contrib"])

    # Aggregate per sample
    S_path = (P.groupby("sample_id")["pathogen_contrib"].sum().rename("S_path").reset_index()
              if not P.empty else pd.DataFrame(columns=["sample_id", "S_path"]))
    S_amr = (G.groupby("sample_id")["arg_contrib"].sum().rename("S_amr").reset_index()
             if not G.empty else pd.DataFrame(columns=["sample_id", "S_amr"]))

    # Interactions
    links, S_int = compute_interaction(df_links, df_p=df_pathogens, df_g=df_args)

    # Combine
    samples = pd.Index(sorted(set(S_path.get('sample_id', pd.Series([]))).union(set(S_amr.get('sample_id', pd.Series([]))).union(set(S_int.get('sample_id', pd.Series([])))))), name='sample_id')
    base = pd.DataFrame({"sample_id": samples})
    base = base.merge(S_path, on="sample_id", how="left")
    base = base.merge(S_amr, on="sample_id", how="left")
    base = base.merge(S_int, on="sample_id", how="left")
    for c in ["S_path", "S_amr", "S_int"]:
        base[c] = base[c].fillna(0.0)
    base["S_raw"] = base[["S_path", "S_amr", "S_int"]].sum(axis=1)

    # Scaling
    base["Score_pctile"], p5, p95 = percentile_scale(base["S_raw"], 5, 95)
    base["Score_sat"], alpha = saturating_scale(base["S_raw"], alpha=None)

    # Bands
    base["Band"] = pd.cut(base["Score_pctile"], bins=RISK_BANDS, labels=RISK_LABELS, include_lowest=True, right=True)

    meta = {
        "percentile_scale": {"p5": float(p5), "p95": float(p95)},
        "saturating_alpha": float(alpha),
        "exposure_multiplier_E": float(EXPOSURE_E),
        "defaults": {
            "pathogen": PATHOGEN_DEFAULT_WEIGHTS,
            "arg": ARG_DEFAULT_WEIGHTS,
            "viability_default_Vp": DEFAULT_VIABILITY_VP,
            "cooccurrence_default_Bpg": DEFAULT_B_PG,
        }
    }

    return base, P, G, links, meta


## Provide input paths (or leave blank to use the synthetic demo)

Update these to point to your CSV/TSV files. If the files are not found, a synthetic dataset will be generated so you can explore the pipeline.

In [ ]:
# Input file paths (relative or absolute). Leave as "" to use synthetic demo.
PATHOGENS_CSV = "pathogens.csv"
ARGS_CSV = "args.csv"
COOCC_CSV = "cooccurrence.csv"  # optional

# Output directory
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Load data or generate a synthetic demo dataset

In [ ]:
need_demo = not (os.path.exists(PATHOGENS_CSV) and os.path.exists(ARGS_CSV))

if need_demo:
    rng = np.random.default_rng(42)
    samples = [f"S{i:02d}" for i in range(1, 7)]
    pathogens = [
        "Campylobacter jejuni",
        "Salmonella enterica",
        "Norovirus",
    ]
    args = [
        "blaNDM",
        "blaCTX-M",
        "qnrS",
    ]
    rows_p = []
    rows_g = []
    for s in samples:
        for p in pathogens:
            # zero-inflated abundances
            if rng.random() < 0.2:
                continue
            abundance = float(np.round(rng.lognormal(mean=2.0, sigma=0.6), 2))  # e.g., genomes/L
            rows_p.append({
                "sample_id": s,
                "pathogen": p,
                "abundance": abundance,
                # Simulate viability evidence occasionally
                "viability": rng.random() < 0.6,
            })
        for g in args:
            if rng.random() < 0.15:
                continue
            abundance = float(np.round(rng.lognormal(mean=2.2, sigma=0.7), 2))  # e.g., copies/L
            rows_g.append({
                "sample_id": s,
                "arg": g,
                "abundance": abundance,
            })
    dfP = pd.DataFrame(rows_p)
    dfG = pd.DataFrame(rows_g)

    # Create a simple co-occurrence set: link blaNDM to Salmonella if both present
    links = []
    for s in samples:
        if ((dfP.sample_id == s) & (dfP.pathogen == "Salmonella enterica")).any() and \
           ((dfG.sample_id == s) & (dfG.arg.str.lower() == "blandm")).any():
            links.append({
                "sample_id": s,
                "pathogen": "Salmonella enterica",
                "arg": "blaNDM",
                "on_plasmid": True,
                # leave B_pg, linked_abundance to be inferred
            })
    dfL = pd.DataFrame(links)
else:
    dfP = load_table(PATHOGENS_CSV, required_cols=["sample_id", "pathogen", "abundance"])
    dfG = load_table(ARGS_CSV, required_cols=["sample_id", "arg", "abundance"])
    dfL = load_table(COOCC_CSV, required_cols=["sample_id", "pathogen", "arg"]) if os.path.exists(COOCC_CSV) else pd.DataFrame()

print("Pathogens rows:", len(dfP))
print("ARG rows:", len(dfG))
print("Co-occurrence links:", len(dfL))

dfP.head(10), dfG.head(10), dfL.head(10)


## Compute scores

In [ ]:
scores, P_annot, G_annot, links_annot, meta = compute_scores(dfP, dfG, dfL)

print("Scaling percentiles (p5, p95):", meta["percentile_scale"]) 
print("Saturating alpha:", meta["saturating_alpha"])

scores.sort_values("Score_pctile", ascending=False).head(10)

## Inspect contributors

The following tables show per-pathogen and per-ARG weights and contributions per sample. You can export these to review top drivers of risk.

In [ ]:
# Top pathogen contributors per sample
P_top = (P_annot.assign(contrib=lambda d: d["pathogen_contrib"]) 
         .sort_values(["sample_id", "contrib"], ascending=[True, False]))

# Top ARG contributors per sample
G_top = (G_annot.assign(contrib=lambda d: d["arg_contrib"]) 
         .sort_values(["sample_id", "contrib"], ascending=[True, False]))

P_top.head(20), G_top.head(20)

## Visualizations

In [ ]:
# Stacked bar: components per sample
comp = scores.melt(id_vars=["sample_id"], value_vars=["S_path", "S_amr", "S_int"],
                   var_name="Component", value_name="Value")
plt.figure(figsize=(12,6))
sns.barplot(data=comp, x="sample_id", y="Value", hue="Component")
plt.title("MRS components per sample (raw units)")
plt.xlabel("Sample")
plt.ylabel("Score contribution (raw)")
plt.legend(title="Component")
plt.tight_layout()
plt.show()

# Scatter: Pathogen vs AMR module
plt.figure(figsize=(6,6))
sns.scatterplot(data=scores, x="S_path", y="S_amr", hue="Score_pctile", palette="viridis", s=120)
plt.title("Pathogen vs AMR module")
plt.xlabel("S_path")
plt.ylabel("S_amr")
plt.tight_layout()
plt.show()

# Final 0–100 score by sample
plt.figure(figsize=(12,4))
sns.barplot(data=scores.sort_values("Score_pctile", ascending=False), x="sample_id", y="Score_pctile", hue="Band", dodge=False)
plt.axhline(20, color='orange', ls='--', lw=1)
plt.axhline(40, color='orange', ls='--', lw=1)
plt.axhline(70, color='red', ls='--', lw=1)
plt.ylim(0, 100)
plt.title("Scaled MRS (0–100)")
plt.xlabel("Sample")
plt.ylabel("Score (percentile-scaled)")
plt.tight_layout()
plt.show()


## Export results

In [ ]:
scores_path = os.path.join(OUTPUT_DIR, "mrs_scores.csv")
P_path = os.path.join(OUTPUT_DIR, "mrs_pathogen_contributors.csv")
G_path = os.path.join(OUTPUT_DIR, "mrs_arg_contributors.csv")
links_path = os.path.join(OUTPUT_DIR, "mrs_interactions.csv")
meta_path = os.path.join(OUTPUT_DIR, "mrs_metadata.json")

scores.to_csv(scores_path, index=False)
P_top.to_csv(P_path, index=False)
G_top.to_csv(G_path, index=False)
links_annot.to_csv(links_path, index=False)
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

print("Wrote:")
print("-", scores_path)
print("-", P_path)
print("-", G_path)
print("-", links_path)
print("-", meta_path)


## Optional: Uncertainty via simple bootstrap or CV assumption

If you have per-feature read counts or confidence metrics, you can bootstrap contributions. Otherwise, you can assume a coefficient of variation (CV) on abundances and propagate uncertainty with Monte Carlo.

In [ ]:
def bootstrap_scores(dfP, dfG, dfL=None, n=200, cv=0.3, random_state=123):
    rng = np.random.default_rng(random_state)
    samples = []
    for i in range(n):
        # Noise abundances with lognormal noise matching CV if counts not provided
        def noise_abund(df, col='abundance'):
            x = df[col].astype(float).values
            if len(x) == 0:
                return df
            # Convert CV to lognormal sigma
            sigma = math.sqrt(math.log(1 + cv**2))
            noise = rng.lognormal(mean=0.0, sigma=sigma, size=len(x))
            df_noisy = df.copy()
            df_noisy[col] = x * noise
            return df_noisy
        Pn = noise_abund(dfP) if not dfP.empty else dfP
        Gn = noise_abund(dfG) if not dfG.empty else dfG
        Ln = dfL.copy() if dfL is not None else dfL
        sc, *_ = compute_scores(Pn, Gn, Ln)
        sc["iter"] = i
        samples.append(sc)
    all_iter = pd.concat(samples, ignore_index=True)
    summary = (all_iter.groupby("sample_id")["S_raw", "Score_pctile", "Score_sat"]
                        .agg([np.mean, lambda x: np.percentile(x, 2.5), lambda x: np.percentile(x, 97.5)]) )
    summary.columns = ["_".join([c[0], c[1] if isinstance(c[1], str) else "ci"]) for c in summary.columns]
    summary = summary.reset_index()
    return all_iter, summary

# Example (commented out to save time):
# all_iter, ci = bootstrap_scores(dfP, dfG, dfL, n=300, cv=0.3)
# ci.head()


## Optional: Simple QMRA illustration (Campylobacter, Salmonella)

This section demonstrates how you might compute per-exposure infection risk for a few well-modeled pathogens using beta-Poisson or exponential dose–response. Use with caution and adapt to your context.

In [ ]:
def qmra_risk(dose, model="beta_poisson", params=None):
    # Returns infection probability for a given dose
    if model == "beta_poisson":
        # P_inf = 1 - 1F1(a, a+b, -dose)
        # Use approximation: P = 1 - (1 + dose/N50 * (2**(1/alpha)-1))**(-alpha)
        alpha = params.get("alpha", 0.145)
        N50 = params.get("N50", 896.0)
        return 1.0 - (1.0 + dose / N50 * (2.0**(1.0/alpha) - 1.0))**(-alpha)
    elif model == "exponential":
        k = params.get("k", 2.18e-3)
        return 1.0 - np.exp(-k * dose)
    else:
        raise ValueError("Unknown model")

# Example: assume ingestion volume V (L) and that 'abundance' is genomes/L ~ dose proxy
V_INGEST_L = 0.1  # 100 mL event

if not dfP.empty:
    dfC = dfP[dfP["pathogen"].str.lower().str.contains("campylobacter")].copy()
    if not dfC.empty:
        dfC["dose"] = dfC["abundance"].astype(float) * V_INGEST_L
        dfC["P_inf"] = dfC["dose"].apply(lambda d: qmra_risk(d, model="beta_poisson", params={"alpha": 0.145, "N50": 896}))
        print("QMRA (Campylobacter, per 100 mL exposure):")
        display(dfC[["sample_id", "pathogen", "abundance", "dose", "P_inf"]].sort_values("P_inf", ascending=False).head(10))

    dfS = dfP[dfP["pathogen"].str.lower().str.contains("salmonella")].copy()
    if not dfS.empty:
        dfS["dose"] = dfS["abundance"].astype(float) * V_INGEST_L
        dfS["P_inf"] = dfS["dose"].apply(lambda d: qmra_risk(d, model="beta_poisson", params={"alpha": 0.3126, "N50": 23600}))
        print("QMRA (Salmonella, per 100 mL exposure):")
        display(dfS[["sample_id", "pathogen", "abundance", "dose", "P_inf"]].sort_values("P_inf", ascending=False).head(10))


## Notes and recommendations

- Prefer absolute quantification (copies or genomes per liter) using spike-ins or qPCR anchoring. If using relative abundances, ensure consistent library sizes and normalization.
- Set clear detection thresholds (e.g., gene coverage, unique reads) upstream of this notebook.
- Customize severity and clinical importance mappings using your own literature-based tables.
- Report module scores (Pathogen, AMR) alongside the composite to improve interpretability.
- Use field blanks/negatives and flag suspicious low-level signals.
- Treat DNA-only detections as potential overestimates for viability; revisit V_p if you have PMA or rRNA evidence.
- The co-occurrence bonus aims to capture mobilized, clinically relevant resistance within pathogens; tune B_pg carefully and document assumptions.
